In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('/home/baijayanta/MY_Documented_Code/tge/')
sys.path.append('/home/baijayanta/MY_Documented_Code/clfuncs/')
import grid as gd
import clfuncs as cfunc

Imported Grid, Import Time : 21/05/26 | 12:08:22 PM
Imported clfuncs, Import Time : 21/05/26 | 12:08:22 PM


In [2]:
# load bining info
BIN = np.load('/home/baijayanta/MY_Documented_Code/tge/bin_info_7200.npz')

# Set the gridding parameters in the TGE code
n1      = 0                  # Starting channel number
n2      = 767                # End channel number
Flag    = True               # Apply actual flagging of data
Umax    = int(BIN['Umax'])   # Baselines greater than this are rejected
FWHM    = int(BIN['FWHM'])   # Primary beam FWHM of the telescope in degrees
f       = float(BIN['f'])    # Tapering parameter value
nstokes = [0,1]              # 0 for XX and 1 for YY (polarizations to grid)
# ===================================================
 
mask = BIN['NI'] >= 0        # mask array
Nbin = int(BIN['Nbin'])      # No of annular bins the whole uv plane has been divided into
ni   = BIN['ni']             # Contains info about which grid point fall into which bin
lval = BIN['lval']           # obtain the ell values

In [3]:
infile   = '/home/MWA/data_cutoff/1000007200.fits'   # fits file name
out_uaps_fits = './simulated/1000007200_uaps.fits'   # uaps fits file name

In [4]:
import os
index = int(os.path.split(infile)[1].split(".")[0][-4:])
print(index)

7200


### $\textrm{Apply to Data to get}\,\,\,\quad e_{\ell}(\nu_a,\nu_b)$

In [5]:
# grid the data first then correlate

nrel = -1 # for data

GV_data = gd.grid(infile, n1, n2, nrel, Umax, FWHM, f, Flag, nstokes)[0]
el      = cfunc.correlate(GV_data[:, mask], ni, Nbin)

# save el file
np.save(f'el_maps_{index}.npy', el)

<<<<<<<<<<< TGE >>>>>>>>>>>>
<< 21/05/26 | 12:08:23 PM >>
Type  : Data
<< 21/05/26 | 12:10:13 PM >>
Elapsed   :   00:01:50.43
<<<<<<<<<<< Done >>>>>>>>>>>
Channels   : 768
Time Taken : 9.010 seconds.



In [6]:
print(el.shape) # (Nbin, nc = n2-n1+1, nc)

(20, 768, 768)


### $\textrm{Apply to UAPS to get}\quad m_{\ell}(\nu_a,\nu_b)$

In [7]:
ml = np.zeros((10, Nbin, (n2-n1+1), (n2-n1+1)))

for reln in range(ml.shape[0]):
    nrel = reln   # for uaps
    print(f'\n<<< Reln = {nrel} >>>\n')
    GV_uaps  = gd.grid(infile, n1, n2, nrel, Umax, FWHM, f, Flag, nstokes)[0]
    ml[reln] = cfunc.correlate(GV_uaps[:, mask], ni, Nbin)

ml = np.mean(ml, axis = 0) # ensemble average

# save ml file
np.save(f'ml_maps_{index}.npy', ml)


<<< Reln = 0 >>>

<<<<<<<<<<< TGE >>>>>>>>>>>>
<< 21/05/26 | 12:10:23 PM >>
Type  : UAPS
Copy  : 0 channel.
<< 21/05/26 | 12:12:11 PM >>
Elapsed   :   00:01:48.10
<<<<<<<<<<< Done >>>>>>>>>>>
Channels   : 768
Time Taken : 7.676 seconds.


<<< Reln = 1 >>>

<<<<<<<<<<< TGE >>>>>>>>>>>>
<< 21/05/26 | 12:12:19 PM >>
Type  : UAPS
Copy  : 1 channel.
<< 21/05/26 | 12:14:06 PM >>
Elapsed   :   00:01:47.35
<<<<<<<<<<< Done >>>>>>>>>>>
Channels   : 768
Time Taken : 7.690 seconds.


<<< Reln = 2 >>>

<<<<<<<<<<< TGE >>>>>>>>>>>>
<< 21/05/26 | 12:14:14 PM >>
Type  : UAPS
Copy  : 2 channel.
<< 21/05/26 | 12:16:02 PM >>
Elapsed   :   00:01:48.04
<<<<<<<<<<< Done >>>>>>>>>>>
Channels   : 768
Time Taken : 7.830 seconds.


<<< Reln = 3 >>>

<<<<<<<<<<< TGE >>>>>>>>>>>>
<< 21/05/26 | 12:16:10 PM >>
Type  : UAPS
Copy  : 3 channel.
<< 21/05/26 | 12:17:59 PM >>
Elapsed   :   00:01:49.02
<<<<<<<<<<< Done >>>>>>>>>>>
Channels   : 768
Time Taken : 7.786 seconds.


<<< Reln = 4 >>>

<<<<<<<<<<< TGE >>>>>>>>>

In [8]:
print(ml.shape) # (Nbin, nc = n2-n1+1, nc)

(20, 768, 768)


### $\textrm{Apply to Noise}\quad n_{\ell}(\nu_a,\nu_b)$

In [10]:
nl = np.zeros((10, Nbin, (n2-n1+1), (n2-n1+1)))

for iter_seed in range(nl.shape[0]):
    seedn = int(index+iter_seed)   # for noise
    print(f'\n<<< Noise = {seedn} >>>\n')
    nrel = -2       # noise
    GV_noise       = gd.grid(infile, n1, n2, nrel, Umax, FWHM, f, Flag, nstokes, seedn)[0]
    nl[iter_seed]  = cfunc.correlate(GV_noise[:, mask], ni, Nbin)

# save nl file
np.save(f'nl_maps_{index}.npy', nl)


<<< Noise = 7200 >>>

<<<<<<<<<<< TGE >>>>>>>>>>>>
<< 21/05/26 | 12:37:32 PM >>
Seed  : 7200
Type  : Noise only
<< 21/05/26 | 12:39:27 PM >>
Elapsed   :   00:01:54.43
<<<<<<<<<<< Done >>>>>>>>>>>
Channels   : 768
Time Taken : 7.632 seconds.


<<< Noise = 7201 >>>

<<<<<<<<<<< TGE >>>>>>>>>>>>
<< 21/05/26 | 12:39:35 PM >>
Seed  : 7201
Type  : Noise only
<< 21/05/26 | 12:41:30 PM >>
Elapsed   :   00:01:54.75
<<<<<<<<<<< Done >>>>>>>>>>>
Channels   : 768
Time Taken : 7.674 seconds.


<<< Noise = 7202 >>>

<<<<<<<<<<< TGE >>>>>>>>>>>>
<< 21/05/26 | 12:41:37 PM >>
Seed  : 7202
Type  : Noise only
<< 21/05/26 | 12:43:32 PM >>
Elapsed   :   00:01:54.87
<<<<<<<<<<< Done >>>>>>>>>>>
Channels   : 768
Time Taken : 7.704 seconds.


<<< Noise = 7203 >>>

<<<<<<<<<<< TGE >>>>>>>>>>>>
<< 21/05/26 | 12:43:40 PM >>
Seed  : 7203
Type  : Noise only
<< 21/05/26 | 12:45:34 PM >>
Elapsed   :   00:01:53.99
<<<<<<<<<<< Done >>>>>>>>>>>
Channels   : 768
Time Taken : 7.683 seconds.


<<< Noise = 7204 >>>

<<<<<

In [11]:
print(nl.shape) # (10, Nbin, nc = n2-n1+1, nc)

(10, 20, 768, 768)


### $\textrm{Calculate MAPS}\quad C_{\ell}(\nu_a,\nu_b) = e_{\ell}(\nu_a,\nu_b)/m_{\ell}(\nu_a,\nu_b)\longrightarrow C_{\ell}(\Delta\nu)$

In [12]:
cl = el/ml
print(cl.shape)

cl_dnu = cfunc.cl_dnu_nua_nub(cl)
print(cl_dnu.shape)

# save cl_dnu file
np.save(f'cldnu_{index}.npy', cl_dnu)

/tmp/ipykernel_208187/3170732372.py:1: RuntimeWarning: invalid value encountered in divide
  cl = el/ml


(20, 768, 768)
(20, 768)


/home/baijayanta/test_corr/clfuncs.py:173: RuntimeWarning: Mean of empty slice
  return np.nanmean(maps1, axis = 2) # maps2


### $\textrm{Calculate MAPS for noise }\quad [C_{\ell}(\nu_a,\nu_b)]_{\rm noise} = n_{\ell}(\nu_a,\nu_b)/m_{\ell}(\nu_a,\nu_b)\longrightarrow [C_{\ell}(\Delta\nu)]_{\rm noise}$

In [13]:
cl_noise = nl/ml
print(cl_noise.shape)

cl_dnu_noise = np.array([cfunc.cl_dnu_nua_nub(cl_noise[ii]) for ii in range(cl_noise.shape[0])])
print(cl_dnu_noise.shape)

# save cl_dnu file for noise
np.save(f'cldnu_noise_{index}.npy', cl_dnu_noise)

/tmp/ipykernel_208187/1091521268.py:1: RuntimeWarning: invalid value encountered in divide
  cl_noise = nl/ml


(10, 20, 768, 768)
(10, 20, 768)
